# Project 4 — Molecular Classification

This notebook is a detailed practice project for molecular classification. It converts molecular structures into RDKit descriptors and Morgan fingerprints, then trains classifiers to predict a binary molecular class.

The main worked example uses the ESOL / Delaney solubility dataset. The original continuous solubility target is converted into a binary label:

- `0` = lower-than-median solubility
- `1` = higher-than-median solubility

This is a classification version of QSAR. The workflow is reusable for toxicity, bioactivity, odour-active prediction or fragrance-family classification when real labels are available.

## Learning objectives

By the end of this notebook, you should be able to:

1. Load a real molecule dataset from a URL or local CSV.
2. Clean SMILES strings and remove invalid or duplicate molecules.
3. Convert a continuous molecular property into a binary classification label.
4. Generate RDKit descriptors and Morgan fingerprints.
5. Train multiple classification models.
6. Evaluate accuracy, precision, recall, F1, ROC-AUC and PR-AUC.
7. Interpret confusion matrices, ROC curves and precision-recall curves.
8. Tune a probability threshold instead of always using `0.5`.
9. Perform error analysis using molecular structures and chemical class tags.
10. Compare random splitting against a scaffold split challenge.
11. Predict labels for new molecules.

## Project map

- Chapter 1 — Imports and configuration
- Chapter 2 — Load and clean ESOL data
- Chapter 3 — Build binary classification labels
- Chapter 4 — RDKit descriptor features
- Chapter 5 — Morgan fingerprint features
- Chapter 6 — Feature-set construction
- Chapter 7 — Model comparison
- Chapter 8 — Cross-validation
- Chapter 9 — Final diagnostics and threshold tuning
- Chapter 10 — Error analysis
- Chapter 11 — Feature importance
- Chapter 12 — Scaffold split challenge
- Chapter 13 — Predict new molecules
- Chapter 14 — Save outputs

## Chapter 1 — Imports and configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, Draw
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
FP_SIZE = 2048
FP_RADIUS = 2
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

## Chapter 2 — Load and clean ESOL data

The ESOL / Delaney dataset contains molecules with measured aqueous solubility values. It is normally used for regression, but here we convert the continuous solubility values into a binary classification label.

The loader tries:

1. A local CSV in common repo paths.
2. The public DeepChem GitHub URL.
3. A small fallback dataset if the URL is unavailable.

In [ ]:
def load_esol_dataset():
    '''Load ESOL / Delaney data from local paths or the public DeepChem URL.'''
    candidate_paths = [
        Path("data/delaney-processed.csv"),
        Path("../data/delaney-processed.csv"),
        Path("data/esol.csv"),
        Path("../data/esol.csv"),
    ]

    for path in candidate_paths:
        if path.exists():
            print(f"Loading local dataset: {path}")
            return pd.read_csv(path)

    url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
    try:
        print("Loading ESOL dataset from DeepChem GitHub URL")
        return pd.read_csv(url)
    except Exception as exc:
        print("Could not load remote ESOL dataset.")
        print("Reason:", exc)
        print("Using a small fallback dataset for code demonstration only.")
        fallback = pd.DataFrame({
            "Compound ID": [
                "methanol", "ethanol", "propanol", "butanol", "benzene", "toluene",
                "phenol", "aniline", "acetic acid", "acetone", "ethyl acetate", "aspirin",
                "caffeine", "ibuprofen", "paracetamol", "glucose"
            ],
            "smiles": [
                "CO", "CCO", "CCCO", "CCCCO", "c1ccccc1", "Cc1ccccc1",
                "Oc1ccccc1", "Nc1ccccc1", "CC(=O)O", "CC(=O)C", "CCOC(=O)C",
                "CC(=O)OC1=CC=CC=C1C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
                "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "CC(=O)NC1=CC=C(O)C=C1",
                "C(C1C(C(C(C(O1)O)O)O)O)O"
            ],
            "measured log solubility in mols per litre": [
                1.57, 1.24, 0.80, 0.30, -1.64, -2.30, -0.04, -0.40,
                1.22, 0.00, -0.30, -2.20, -0.55, -3.50, -1.20, 1.60
            ],
        })
        return fallback

raw_df = load_esol_dataset()
print("Raw shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
raw_df.head()

In [ ]:
def prepare_esol_columns(raw_df):
    '''Standardise common ESOL column names.'''
    df = raw_df.copy()
    rename_map = {
        "Compound ID": "name",
        "compound_id": "name",
        "Name": "name",
        "SMILES": "smiles",
        "canonical_smiles": "smiles",
        "measured log solubility in mols per litre": "target_logS",
        "logS": "target_logS",
        "target": "target_logS",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    required = ["name", "smiles", "target_logS"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Available columns: {df.columns.tolist()}")

    df = df[required].dropna().copy()
    df["name"] = df["name"].astype(str)
    df["smiles"] = df["smiles"].astype(str)
    df["target_logS"] = pd.to_numeric(df["target_logS"], errors="coerce")
    df = df.dropna(subset=["target_logS"]).reset_index(drop=True)
    return df

df = prepare_esol_columns(raw_df)
print("Prepared shape:", df.shape)
df.head()

In [ ]:
def smiles_to_mol(smiles):
    '''Convert a SMILES string to an RDKit molecule object.'''
    if pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None


def canonicalise_mol(mol):
    '''Return canonical SMILES for a valid RDKit molecule.'''
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


df["mol"] = df["smiles"].apply(smiles_to_mol)
invalid_count = int(df["mol"].isna().sum())
df = df[df["mol"].notna()].copy()

df["canonical_smiles"] = df["mol"].apply(canonicalise_mol)

before_dedup = len(df)
df = df.drop_duplicates(subset="canonical_smiles").reset_index(drop=True)
after_dedup = len(df)

print("Invalid SMILES removed:", invalid_count)
print("Duplicates removed:", before_dedup - after_dedup)
print("Clean shape:", df.shape)
df[["name", "smiles", "canonical_smiles", "target_logS"]].head()

## Chapter 3 — Build binary classification labels

A regression target such as `logS` can be converted into a classification task by defining a threshold.

In this notebook, the threshold is the median measured log solubility in the dataset.

- `label = 1`: solubility above median
- `label = 0`: solubility at or below median

This creates a balanced demonstration task. For real applications, the threshold should come from domain criteria, assay definitions or product requirements.

In [ ]:
threshold = df["target_logS"].median()
df["label"] = (df["target_logS"] > threshold).astype(int)

print("Median logS threshold:", threshold)
print("Label counts:")
print(df["label"].value_counts().sort_index())
print("Label proportions:")
print(df["label"].value_counts(normalize=True).sort_index())

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(df["target_logS"], bins=30, alpha=0.8)
plt.axvline(threshold, linestyle="--", label=f"Median threshold = {threshold:.2f}")
plt.xlabel("Measured logS")
plt.ylabel("Number of molecules")
plt.title("ESOL target distribution and classification threshold")
plt.legend()
plt.show()

In [ ]:
label_summary = df.groupby("label")["target_logS"].agg(["count", "mean", "std", "min", "median", "max"])
label_summary

## Chapter 4 — RDKit descriptor features

RDKit descriptors are human-interpretable molecular features. They are useful baselines because they represent global properties such as molecular size, hydrophobicity, polarity and hydrogen-bonding capacity.

In [ ]:
def calculate_descriptors(mol):
    '''Calculate a compact descriptor panel for classification practice.'''
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "MolMR": Crippen.MolMR(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "HeavyAtomCount": Lipinski.HeavyAtomCount(mol),
        "RingCount": Lipinski.RingCount(mol),
        "AromaticRingCount": rdMolDescriptors.CalcNumAromaticRings(mol),
        "AliphaticRingCount": rdMolDescriptors.CalcNumAliphaticRings(mol),
        "SaturatedRingCount": rdMolDescriptors.CalcNumSaturatedRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "BertzCT": Descriptors.BertzCT(mol),
        "BalabanJ": Descriptors.BalabanJ(mol),
        "LabuteASA": rdMolDescriptors.CalcLabuteASA(mol),
    }


desc_df = pd.DataFrame([calculate_descriptors(mol) for mol in df["mol"]])

print("Descriptor shape:", desc_df.shape)
desc_df.head()

In [ ]:
# Quick descriptor summary by class.
desc_with_label = pd.concat([desc_df, df[["label", "target_logS"]].reset_index(drop=True)], axis=1)
class_descriptor_summary = desc_with_label.groupby("label").agg(["mean", "std"])
class_descriptor_summary[["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds"]]

In [ ]:
selected_descriptors = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds"]

for col in selected_descriptors:
    plt.figure(figsize=(7, 4))
    for label in [0, 1]:
        values = desc_with_label.loc[desc_with_label["label"] == label, col]
        plt.hist(values, bins=25, alpha=0.55, label=f"Class {label}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"{col} distribution by class")
    plt.legend()
    plt.show()

## Chapter 5 — Morgan fingerprint features

Morgan fingerprints encode local molecular substructures into a fixed-length bit vector. They are useful for QSAR classification because they capture fragments and atom environments that may not be represented by global descriptors.

In [ ]:
morgan_generator = GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)


def mol_to_morgan_fp(mol):
    '''Convert an RDKit molecule to a Morgan fingerprint array.'''
    fp = morgan_generator.GetFingerprint(mol)
    return np.asarray(fp, dtype=np.float32)


fp_array = np.stack(df["mol"].apply(mol_to_morgan_fp).values)
fp_df_full = pd.DataFrame(fp_array, columns=[f"Morgan_{i}" for i in range(FP_SIZE)])

bit_frequency = fp_df_full.mean(axis=0)
active_fp_cols = bit_frequency[bit_frequency > 0].index.tolist()
fp_df = fp_df_full[active_fp_cols].copy()

print("Full fingerprint shape:", fp_df_full.shape)
print("Active fingerprint shape:", fp_df.shape)
print("Inactive bits removed:", FP_SIZE - len(active_fp_cols))
print("Mean fingerprint density:", fp_array.mean())
fp_df.head()

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(fp_df_full.sum(axis=1), bins=30, alpha=0.8)
plt.xlabel("Number of active Morgan bits per molecule")
plt.ylabel("Number of molecules")
plt.title("Morgan fingerprint density distribution")
plt.show()

## Chapter 6 — Feature-set construction

We compare three feature sets:

1. Descriptors only
2. Morgan fingerprints only
3. Descriptors + Morgan fingerprints

This is useful because descriptor models are more interpretable, while fingerprint models usually capture richer structural information.

In [ ]:
X_desc = desc_df.reset_index(drop=True)
X_fp = fp_df.reset_index(drop=True)
X_combined = pd.concat([X_desc, X_fp], axis=1)
y = df["label"].astype(int).reset_index(drop=True)

feature_sets = {
    "Descriptors only": X_desc,
    "Morgan only": X_fp,
    "Descriptors + Morgan": X_combined,
}

for name, X in feature_sets.items():
    print(f"{name}: {X.shape}")

print("Target shape:", y.shape)
print("Target counts:")
print(y.value_counts().sort_index())

## Chapter 7 — Model comparison

For classification, use classification metrics instead of regression metrics.

Important metrics:

- Accuracy: overall fraction of correct predictions.
- Precision: among predicted positives, how many are true positives.
- Recall: among actual positives, how many are detected.
- F1: harmonic mean of precision and recall.
- ROC-AUC: how well the model ranks class 1 above class 0 across thresholds.
- PR-AUC: useful when the positive class is rare.

In [ ]:
def get_classification_models():
    '''Return a small model zoo for molecular classification.'''
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "SVM RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        "kNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=7)),
        ]),
    }

In [ ]:
def classification_metrics(y_true, y_pred, y_prob):
    '''Calculate common binary classification metrics.'''
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }


def evaluate_feature_set(X, y, feature_set_name, test_size=0.25):
    '''Train all models on one feature set and return metrics and fitted objects.'''
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    rows = []
    fitted = {}
    predictions = {}

    for model_name, model in get_classification_models().items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        row = {
            "feature_set": feature_set_name,
            "model": model_name,
            **classification_metrics(y_test, y_pred, y_prob),
        }
        rows.append(row)
        fitted[model_name] = model
        predictions[model_name] = {"pred": y_pred, "prob": y_prob}

    split_data = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }

    return pd.DataFrame(rows), fitted, predictions, split_data

In [ ]:
all_result_tables = []
all_fitted_models = {}
all_predictions = {}
all_splits = {}

for feature_set_name, X in feature_sets.items():
    result_df, fitted, preds, split_data = evaluate_feature_set(X, y, feature_set_name)
    all_result_tables.append(result_df)
    all_fitted_models[feature_set_name] = fitted
    all_predictions[feature_set_name] = preds
    all_splits[feature_set_name] = split_data

results_df = pd.concat(all_result_tables, ignore_index=True)
results_df = results_df.sort_values(["f1", "roc_auc", "pr_auc"], ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = results_df.sort_values("roc_auc", ascending=True)
labels = plot_df["feature_set"] + " | " + plot_df["model"]
plt.barh(labels, plot_df["roc_auc"])
plt.xlabel("ROC-AUC")
plt.title("Model comparison by ROC-AUC")
plt.xlim(0.0, 1.0)
plt.show()

## Chapter 8 — Cross-validation

A single train/test split can be unstable. Cross-validation gives a better estimate of model performance, especially for small or medium-sized chemistry datasets.

In [ ]:
min_class_count = int(y.value_counts().min())
n_splits = min(5, min_class_count)
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for feature_set_name, X in feature_sets.items():
    for model_name, model in get_classification_models().items():
        f1_scores = cross_val_score(model, X, y, cv=cv, scoring="f1")
        roc_scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
        cv_rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "CV_F1_mean": f1_scores.mean(),
            "CV_F1_std": f1_scores.std(),
            "CV_ROC_AUC_mean": roc_scores.mean(),
            "CV_ROC_AUC_std": roc_scores.std(),
        })

cv_results_df = pd.DataFrame(cv_rows)
cv_results_df = cv_results_df.sort_values(["CV_F1_mean", "CV_ROC_AUC_mean"], ascending=False).reset_index(drop=True)
cv_results_df

In [ ]:
best_cv_row = cv_results_df.iloc[0]
best_feature_set_name = best_cv_row["feature_set"]
best_model_name = best_cv_row["model"]

print("Best feature set:", best_feature_set_name)
print("Best model:", best_model_name)
print("Best CV F1:", best_cv_row["CV_F1_mean"])
print("Best CV ROC-AUC:", best_cv_row["CV_ROC_AUC_mean"])

X_best = feature_sets[best_feature_set_name]

## Chapter 9 — Final diagnostics and threshold tuning

The default probability threshold is `0.5`, but it is often not optimal. For many molecular screening tasks, you may prefer higher recall or higher precision.

This section tunes the threshold on a validation set, then evaluates the chosen threshold on a held-out test set.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_best,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

final_model = clone(get_classification_models()[best_model_name])
final_model.fit(X_train, y_train)

valid_prob = final_model.predict_proba(X_valid)[:, 1]
test_prob = final_model.predict_proba(X_test)[:, 1]

valid_pred_default = (valid_prob >= 0.5).astype(int)
test_pred_default = (test_prob >= 0.5).astype(int)

print("Default threshold validation metrics:")
print(classification_report(y_valid, valid_pred_default, zero_division=0))
print("Default threshold test metrics:")
print(classification_report(y_test, test_pred_default, zero_division=0))

In [ ]:
threshold_rows = []
thresholds = np.arange(0.05, 0.96, 0.01)

for threshold_value in thresholds:
    pred = (valid_prob >= threshold_value).astype(int)
    threshold_rows.append({
        "threshold": threshold_value,
        "accuracy": accuracy_score(y_valid, pred),
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "f1": f1_score(y_valid, pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df = threshold_df.sort_values("f1", ascending=False).reset_index(drop=True)
best_threshold = float(threshold_df.loc[0, "threshold"])

print("Best validation threshold:", best_threshold)
threshold_df.head(10)

In [ ]:
test_pred_tuned = (test_prob >= best_threshold).astype(int)

print("Tuned-threshold test metrics:")
print(classification_report(y_test, test_pred_tuned, zero_division=0))

final_metrics_default = classification_metrics(y_test, test_pred_default, test_prob)
final_metrics_tuned = classification_metrics(y_test, test_pred_tuned, test_prob)

pd.DataFrame([
    {"threshold": 0.5, **final_metrics_default},
    {"threshold": best_threshold, **final_metrics_tuned},
])

In [ ]:
cm = confusion_matrix(y_test, test_pred_tuned)
ConfusionMatrixDisplay(cm, display_labels=["Class 0", "Class 1"]).plot()
plt.title(f"Confusion matrix — {best_model_name}, tuned threshold")
plt.show()

RocCurveDisplay.from_predictions(y_test, test_prob)
plt.title(f"ROC curve — {best_model_name}")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, test_prob)
plt.title(f"Precision-recall curve — {best_model_name}")
plt.show()

## Chapter 10 — Error analysis

Error analysis is essential in chemistry ML. It helps identify whether the model fails on particular chemical families, outliers or unusual scaffolds.

In [ ]:
# Build a compact table for the final test split.
test_indices = X_test.index if hasattr(X_test, "index") else pd.Index(range(len(X_test)))

test_results = df.loc[test_indices, ["name", "canonical_smiles", "target_logS", "label", "mol"]].copy()
test_results = test_results.reset_index(drop=True)
test_results["predicted_label"] = test_pred_tuned
test_results["probability_class_1"] = test_prob
test_results["correct"] = test_results["label"] == test_results["predicted_label"]
test_results["error_type"] = np.select(
    [
        (test_results["label"] == 0) & (test_results["predicted_label"] == 1),
        (test_results["label"] == 1) & (test_results["predicted_label"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

test_results.head()

In [ ]:
error_table = test_results[test_results["correct"] == False].copy()
error_table = error_table.sort_values("probability_class_1", ascending=False)
print("Number of test errors:", len(error_table))
error_table[["name", "canonical_smiles", "target_logS", "label", "predicted_label", "probability_class_1", "error_type"]].head(20)

In [ ]:
# Visualise a small set of misclassified molecules.
error_mols = [mol for mol in error_table["mol"].head(12) if mol is not None]
error_legends = [
    f"{row.name}: true={row.label}, pred={row.predicted_label}, p={row.probability_class_1:.2f}"
    for row in error_table.head(12).itertuples()
]

if error_mols:
    img = Draw.MolsToGridImage(
        error_mols,
        molsPerRow=4,
        subImgSize=(250, 220),
        legends=error_legends,
    )
    display(img)
else:
    print("No misclassified molecules to display.")

In [ ]:
def tag_chemical_class(mol):
    '''Assign broad chemical class tags using simple SMARTS patterns.'''
    patterns = {
        "carboxylic_acid": "C(=O)[OX2H1]",
        "ester": "C(=O)O[#6]",
        "amide": "C(=O)N",
        "amine": "[NX3;H2,H1,H0;!$(NC=O)]",
        "alcohol_or_phenol": "[OX2H]",
        "aromatic": "a",
        "alkene": "C=C",
        "halogenated": "[F,Cl,Br,I]",
        "nitrile": "C#N",
        "sulfone_or_sulfoxide": "S(=O)",
    }
    tags = []
    for name, smarts in patterns.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt is not None and mol.HasSubstructMatch(patt):
            tags.append(name)
    return ";".join(tags) if tags else "other"


test_results["chemical_class_tags"] = test_results["mol"].apply(tag_chemical_class)

def primary_tag(tag_string):
    return tag_string.split(";")[0]

test_results["primary_class"] = test_results["chemical_class_tags"].apply(primary_tag)

class_error_summary = (
    test_results
    .groupby("primary_class")
    .agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        mean_probability_class_1=("probability_class_1", "mean"),
        mean_target_logS=("target_logS", "mean"),
    )
    .sort_values("n", ascending=False)
)
class_error_summary

## Chapter 11 — Feature importance

Tree-based classifiers expose `feature_importances_`. Linear models expose coefficients. These are useful for rough interpretation, but fingerprint bits are less directly interpretable than descriptor names.

In [ ]:
def extract_final_estimator(model):
    '''Return final estimator from a pipeline or the model itself.'''
    if hasattr(model, "named_steps"):
        return model.named_steps.get("model")
    return model


def show_model_importance(model, feature_names, top_n=25):
    '''Show feature importances or coefficients when available.'''
    estimator = extract_final_estimator(model)

    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        label = "importance"
    elif hasattr(estimator, "coef_"):
        values = np.ravel(estimator.coef_)
        label = "coefficient"
    else:
        print("This model does not expose feature importances or coefficients.")
        return None

    importance_df = pd.DataFrame({
        "feature": list(feature_names),
        label: values,
        "abs_value": np.abs(values),
    }).sort_values("abs_value", ascending=False)

    display(importance_df.head(top_n))

    plot_df = importance_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(8, 7))
    plt.barh(plot_df["feature"], plot_df[label])
    plt.xlabel(label)
    plt.title(f"Top {top_n} model features")
    plt.show()

    return importance_df

importance_df = show_model_importance(final_model, X_best.columns, top_n=25)

In [ ]:
# If the best model uses descriptor features, inspect descriptor-level separation.
descriptor_separation = []
for col in desc_df.columns:
    class0 = desc_with_label.loc[desc_with_label["label"] == 0, col]
    class1 = desc_with_label.loc[desc_with_label["label"] == 1, col]
    descriptor_separation.append({
        "descriptor": col,
        "class0_mean": class0.mean(),
        "class1_mean": class1.mean(),
        "difference_class1_minus_class0": class1.mean() - class0.mean(),
    })

descriptor_separation_df = pd.DataFrame(descriptor_separation)
descriptor_separation_df = descriptor_separation_df.reindex(
    descriptor_separation_df["difference_class1_minus_class0"].abs().sort_values(ascending=False).index
)
descriptor_separation_df.head(20)

## Chapter 12 — Scaffold split challenge

A random split may place very similar molecules in both train and test sets. A scaffold split is harder because the test set contains different core structures. This is closer to real molecular generalisation.

In [ ]:
def mol_to_scaffold_smiles(mol):
    '''Return Murcko scaffold SMILES for a molecule.'''
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold, canonical=True)
    except Exception:
        return ""


df["murcko_scaffold"] = df["mol"].apply(mol_to_scaffold_smiles)
print("Number of unique scaffolds:", df["murcko_scaffold"].nunique())
df[["name", "canonical_smiles", "murcko_scaffold"]].head()

In [ ]:
def scaffold_train_test_indices(df, test_fraction=0.20):
    '''Create a simple deterministic scaffold split.'''
    scaffold_to_indices = {}
    for idx, scaffold in enumerate(df["murcko_scaffold"]):
        scaffold_to_indices.setdefault(scaffold, []).append(idx)

    scaffold_groups = sorted(scaffold_to_indices.values(), key=len, reverse=True)
    target_test_size = int(round(len(df) * test_fraction))

    train_indices = []
    test_indices = []

    for group in scaffold_groups:
        if len(test_indices) < target_test_size:
            test_indices.extend(group)
        else:
            train_indices.extend(group)

    return np.array(train_indices), np.array(test_indices)


train_idx, test_idx = scaffold_train_test_indices(df, test_fraction=0.20)

X_scaffold = X_best.reset_index(drop=True)
y_scaffold = y.reset_index(drop=True)

X_train_scaf = X_scaffold.iloc[train_idx]
y_train_scaf = y_scaffold.iloc[train_idx]
X_test_scaf = X_scaffold.iloc[test_idx]
y_test_scaf = y_scaffold.iloc[test_idx]

print("Scaffold train size:", X_train_scaf.shape)
print("Scaffold test size:", X_test_scaf.shape)
print("Train label counts:", np.bincount(y_train_scaf))
print("Test label counts:", np.bincount(y_test_scaf))

In [ ]:
scaffold_model = clone(get_classification_models()[best_model_name])
scaffold_model.fit(X_train_scaf, y_train_scaf)

scaf_prob = scaffold_model.predict_proba(X_test_scaf)[:, 1]
scaf_pred = (scaf_prob >= 0.5).astype(int)

scaffold_metrics = classification_metrics(y_test_scaf, scaf_pred, scaf_prob)
print("Scaffold split metrics:")
print(scaffold_metrics)
print(classification_report(y_test_scaf, scaf_pred, zero_division=0))

## Chapter 13 — Predict new molecules

This section predicts whether new molecules are above or below the ESOL median solubility threshold.

The same feature pipeline must be used for training and prediction.

In [ ]:
new_molecules = pd.DataFrame({
    "name": [
        "benzyl acetate",
        "ambroxide",
        "glucose",
        "limonene",
        "caffeine",
        "ibuprofen",
        "vanillin",
    ],
    "smiles": [
        "CC(=O)OCC1=CC=CC=C1",
        "CC1(C)CCCC2(C)C1CCC(O2)C(C)(C)O",
        "C(C1C(C(C(C(O1)O)O)O)O)O",
        "CC1=CCC(CC1)C(=C)C",
        "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
        "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
        "COC1=C(C=CC(=C1)C=O)O",
    ],
})

new_molecules["mol"] = new_molecules["smiles"].apply(smiles_to_mol)
new_molecules = new_molecules[new_molecules["mol"].notna()].reset_index(drop=True)
new_molecules["canonical_smiles"] = new_molecules["mol"].apply(canonicalise_mol)

new_desc = pd.DataFrame([calculate_descriptors(mol) for mol in new_molecules["mol"]])
new_fp_full = pd.DataFrame(
    np.stack(new_molecules["mol"].apply(mol_to_morgan_fp).values),
    columns=[f"Morgan_{i}" for i in range(FP_SIZE)],
)
new_fp = new_fp_full.reindex(columns=active_fp_cols, fill_value=0)

if best_feature_set_name == "Descriptors only":
    X_new = new_desc.reindex(columns=X_best.columns, fill_value=0)
elif best_feature_set_name == "Morgan only":
    X_new = new_fp.reindex(columns=X_best.columns, fill_value=0)
else:
    X_new = pd.concat([new_desc, new_fp], axis=1).reindex(columns=X_best.columns, fill_value=0)

new_prob = final_model.predict_proba(X_new)[:, 1]
new_pred = (new_prob >= best_threshold).astype(int)

new_molecules["predicted_label"] = new_pred
new_molecules["probability_class_1"] = new_prob
new_molecules["interpretation"] = np.where(
    new_molecules["predicted_label"] == 1,
    "predicted above median solubility",
    "predicted below median solubility",
)

new_molecules[["name", "canonical_smiles", "predicted_label", "probability_class_1", "interpretation"]]

In [ ]:
new_mols = [mol for mol in new_molecules["mol"]]
new_legends = [
    f"{row.name}: class={row.predicted_label}, p={row.probability_class_1:.2f}"
    for row in new_molecules.itertuples()
]

img = Draw.MolsToGridImage(new_mols, molsPerRow=4, subImgSize=(250, 220), legends=new_legends)
display(img)

## Chapter 14 — Save outputs

The output tables are useful for GitHub examples, model reports and later comparison with Projects 3 and 5.

In [ ]:
results_df.to_csv(OUTPUT_DIR / "project4_model_comparison.csv", index=False)
cv_results_df.to_csv(OUTPUT_DIR / "project4_cross_validation.csv", index=False)
threshold_df.to_csv(OUTPUT_DIR / "project4_threshold_tuning.csv", index=False)
test_results.drop(columns=["mol"]).to_csv(OUTPUT_DIR / "project4_test_predictions.csv", index=False)
new_molecules.drop(columns=["mol"]).to_csv(OUTPUT_DIR / "project4_new_molecule_predictions.csv", index=False)

print("Saved output files to:", OUTPUT_DIR.resolve())

## Final notes

This notebook demonstrates molecular classification using a real measured dataset converted into a binary label. The same workflow can be reused for:

- toxic vs non-toxic
- active vs inactive
- soluble vs insoluble
- odour-active vs non-odour-active
- fragrance-like vs non-fragrance-like
- multi-class fragrance family prediction

For real perfume or toxicity modelling, replace the median-split ESOL label with experimentally curated class labels.